# P10: CTC y alineamiento

**Contenido del notebook**

1. Completar la condición que decide cuándo agregar una etiqueta al texto. Probar con los caminos provistos.
2. Completar las cuatro líneas: la inicialización, la condición de salto, el cálculo de $\alpha_t(s)$ y la suma final. Comparar la tabla y la probabilidad obtenidas con las del ejercicio 2.

3. Comparar la pérdida calculada con el forward con la que devuelve `CTCLoss` en los tres casos provistos. Explicá qué contiene `log_probs` y qué se pasa como argumento `targets`.

4. Para
   $$
   P_{\mathrm b} =
   \begin{bmatrix}
   0{,}40 & 0{,}50 & 0{,}10\\
   0{,}40 & 0{,}15 & 0{,}45
   \end{bmatrix}
   $$
obtener el camino *greedy*, su probabilidad y el texto que resulta al colapsarlo. Después, emplear la enumeración provista para hallar el texto más probable. Escribir sólo los caminos que producen ese texto y explicar por qué la suma de sus probabilidades supera la del camino *greedy*.

Usamos columnas en el orden `("<blank>", "c", "a")` y una matriz de probabilidades ya calculada. En Python, los índices $t$ y $s$ empiezan en 0. El índice de `"<blank>"` es 0.

In [1]:
from collections import defaultdict
from itertools import product

import numpy as np
import torch

In [2]:
tokens = ["<blank>", "c", "a"]
blank = 0
P = np.array([[0.2, 0.7, 0.1], [0.3, 0.2, 0.5], [0.4, 0.1, 0.5]], dtype=np.float64)
ids_transcripcion = [1, 2]

def dar_formato(seq):
    return '(' + ', '.join(f'"{tokens[v]}"' for v in seq) + ')'

def extender(ids_transcripcion, blank=0):
    ext = np.full(2 * len(ids_transcripcion) + 1, blank, dtype=int)
    ext[1::2] = ids_transcripcion
    return ext

ext = extender(ids_transcripcion, blank)
print('Transcripción extendida:', dar_formato(ext))

Transcripción extendida: ("<blank>", "c", "<blank>", "a", "<blank>")


## 1. Colapso

Completar la condición para agregar una etiqueta. Observar qué ocurre con `("c", "<blank>", "c")` frente a `("c", "c")`.
Comentarios: "Agregar etiqueta" se refiere a "Cómo elijo qué etiqueta pasa a la salida y cuál no"

In [3]:
def colapsar(camino, blank=0):
    texto = []
    anterior = None
    for v in camino:
        agregar = (v != anterior) and (v != blank) ### <-- COMPLETAR
        if agregar:
            texto.append(v)
        anterior = v
    return texto

for camino in [[0, 1, 1, 0, 2], [1, 0, 1, 2, 0]]:
    print(dar_formato(camino), '->', dar_formato(colapsar(camino, blank)))

("<blank>", "c", "c", "<blank>", "a") -> ("c", "a")
("c", "<blank>", "c", "a", "<blank>") -> ("c", "c", "a")


## 2. Forward de CTC

Completar la inicialización, la condición de salto, la probabilidad de llegada y la suma final. Se asume $T\geq 1$ y una transcripción no vacía.

In [4]:
def ctc_forward(P, ext, blank=0):
    T = P.shape[0]
    S = len(ext)
    alpha = np.zeros((T, S), dtype=np.float64)
    alpha[0, :2] = P[0, ext[:2]] ### <-- COMPLETAR
    for t in range(1, T):
        for s in range(S):
            total = alpha[t - 1, s]
            if s >= 1:
                total += alpha[t - 1, s - 1]
            salto = (s >= 2) and (ext[s] != blank) and (ext[s] != ext[s-2])  ### <-- COMPLETAR
            if salto:
                total += alpha[t - 1, s - 2]
            alpha[t, s] = P[t, ext[s]] * total  ### <-- COMPLETAR
    prob = alpha[-1, -1] + alpha[-1, -2] ### <-- COMPLETAR
    return alpha, prob

alpha, prob = ctc_forward(P, ext, blank)
print(np.round(alpha, 3))
print(f'P(transcripcion|audio) = {prob:.6f}; pérdida = {-np.log(prob):.6f}')

[[0.2   0.7   0.    0.    0.   ]
 [0.06  0.18  0.21  0.35  0.   ]
 [0.024 0.024 0.156 0.37  0.14 ]]
P(transcripcion|audio) = 0.510000; pérdida = 0.673345


## 3. Contraste con PyTorch

La biblioteca espera log-probabilidades con dimensiones tiempo, batch, clases. Pasamos la transcripción original, es decir, sin blanks. Con `zero_infinity=False` una transcripción imposible da una pérdida $+\infty$.

In [5]:
def perdida_torch(P, target):
    log_probs = torch.from_numpy(np.log(P)).unsqueeze(1)
    etiquetas = torch.tensor(target, dtype=torch.long)
    long_audio = torch.tensor([P.shape[0]], dtype=torch.long)
    long_texto = torch.tensor([len(target)], dtype=torch.long)
    criterio = torch.nn.CTCLoss(blank=blank, reduction='none', zero_infinity=False)
    return criterio(log_probs, etiquetas, long_audio, long_texto).item()

for matriz, texto in [(P, ids_transcripcion), (P, [1, 1]), (P[:2], [1, 1])]:
    _, prob_texto = ctc_forward(matriz, extender(texto, blank), blank)
    loss = -np.log(prob_texto) if prob_texto > 0 else np.inf
    ref = perdida_torch(matriz, texto)
    print(f'T={len(matriz)}, transcripcion={dar_formato(texto)}: '
          f'P={prob_texto:.6f}; forward={loss:.6f}; PyTorch={ref:.6f}')
    assert np.isclose(loss, ref)

T=3, transcripcion=("c", "a"): P=0.510000; forward=0.673345; PyTorch=0.673345
T=3, transcripcion=("c", "c"): P=0.021000; forward=3.863233; PyTorch=3.863233
T=2, transcripcion=("c", "c"): P=0.000000; forward=inf; PyTorch=inf


## 4. Greedy frente a suma por texto

Completar el camino greedy. Para $P_{\rm b}$ hay nueve caminos y se los puede enumerar. ¿Qué caminos contribuyen al texto *ganador*?

In [11]:
def greedy(P, blank=0):
    camino = np.argmax(P, axis=1).tolist()  ### <-- COMPLETAR
    return camino, colapsar(camino, blank)

def masas_por_texto(P):
    masas = defaultdict(float)
    T, n_clases = P.shape
    for camino in product(range(n_clases), repeat=T):
        texto = tuple(colapsar(camino, blank))
        masas[texto] += float(np.prod(P[np.arange(T), camino]))
    return sorted(masas.items(), key=lambda par: par[1], reverse=True)

P_b = np.array([[0.40, 0.50, 0.10], [0.40, 0.15, 0.45]])
camino, salida = greedy(P_b, blank)
print('Greedy:', dar_formato(camino), '->', dar_formato(salida))
print('Masa del camino greedy:', np.prod(P_b[np.arange(len(P_b)), camino]))
print("\nAclaración: Los siguientes caminos son colapsados!")
for texto, masa in masas_por_texto(P_b):
    print(dar_formato(texto), f'{masa:.3f}')

Greedy: ("c", "a") -> ("c", "a")
Masa del camino greedy: 0.225

Aclaración: Los siguientes caminos son colapsados!
("c") 0.335
("a") 0.265
("c", "a") 0.225
() 0.160
("a", "c") 0.015
